In [1]:
import os, json
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from langchain_community.document_loaders import PyPDFLoader

PDF_PATH = "telecom_guide.pdf"

loader = PyPDFLoader(PDF_PATH)
pages = loader.load()

print(f"Loaded {len(pages)} pages from the PDF.")
print("\n--- Fist page preview (first 500 chars) ---")
print(pages[0].page_content[:500])

/var/folders/9q/b_290j4s44s387j_9hglwv640000gn/T/ipykernel_43412/849537456.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


Loaded 9 pages from the PDF.

--- Fist page preview (first 500 chars) ---
Telecom Technical Reference Guide  - Internal Use Only
Telecom Technical
Reference Guide
Customer Care & Network Operations Edition
Version 3.2  |  Covers 2G / 3G / 4G LTE / 5G
Page 1


In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=600,       # ~150 words per chunk
    chunk_overlap=100,    # overlap keeps context at boundaries
    separators=["\n\n", "\n", ".", " "],  # tries paragraph → line → sentence → word
)

chunks = splitter.split_documents(pages)

In [4]:
print(chunks)

[Document(metadata={'producer': 'PyPDF', 'creator': 'PyPDF', 'creationdate': '2026-04-30T14:16:24+00:00', 'source': 'telecom_guide.pdf', 'total_pages': 9, 'page': 0, 'page_label': '1'}, page_content='Telecom Technical Reference Guide  - Internal Use Only\nTelecom Technical\nReference Guide\nCustomer Care & Network Operations Edition\nVersion 3.2  |  Covers 2G / 3G / 4G LTE / 5G\nPage 1'), Document(metadata={'producer': 'PyPDF', 'creator': 'PyPDF', 'creationdate': '2026-04-30T14:16:24+00:00', 'source': 'telecom_guide.pdf', 'total_pages': 9, 'page': 1, 'page_label': '2'}, page_content='Telecom Technical Reference Guide  - Internal Use Only\n1. Introduction to Mobile Networks\nMobile networks have evolved through several generations, each offering significant improvements in speed,\ncapacity, and capability.\n2G (GSM) networks introduced digital voice and basic data services such as SMS. Data speeds were limited to\naround 50 kbps, sufficient only for text messaging and simple email.\n3G 

In [5]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vector_store = Chroma.from_documents(chunks, embeddings)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [6]:
print(f"Vecor store ready. {vector_store._collection.count()} vectors stored.")

Vecor store ready. 37 vectors stored.


In [7]:
retriever = vector_store.as_retriever(search_kwargs={"k": 3})

In [8]:
test_query = "What is the VoLTE and how does it improve call quality?"
retrieved = retriever.invoke(test_query)

In [9]:
print(retrieved)

[Document(id='68f3e2c6-5615-4689-80d4-0f145eff06bd', metadata={'page_label': '7', 'creator': 'PyPDF', 'producer': 'PyPDF', 'page': 6, 'source': 'telecom_guide.pdf', 'total_pages': 9, 'creationdate': '2026-04-30T14:16:24+00:00'}, page_content='Telecom Technical Reference Guide  - Internal Use Only\n6. VoLTE, VoWiFi, and Advanced Voice Services\nVoice over LTE (VoLTE) and Voice over Wi-Fi (VoWiFi) are IP-based voice technologies that replace the legacy\ncircuit-switched voice channel used in 2G and 3G networks.\nVoLTE: With VoLTE, voice calls are transmitted as data packets over the LTE network using the IMS (IP\nMultimedia Subsystem) core. Benefits include HD voice quality (wideband audio at 16 kHz versus the 3.4 kHz of\nlegacy calls), faster call setup times (under 2 seconds versus 6-8 seconds on 3G), and the ability to use data and'), Document(id='26b23a44-6787-4236-a4cb-61fda9acedf0', metadata={'page_label': '7', 'creationdate': '2026-04-30T14:16:24+00:00', 'creator': 'PyPDF', 'total

In [10]:
for i, doc in enumerate(retrieved, 1):
    print(f"--- Chunk {i} ---")
    print(doc.page_content[:300])
    print()

--- Chunk 1 ---
Telecom Technical Reference Guide  - Internal Use Only
6. VoLTE, VoWiFi, and Advanced Voice Services
Voice over LTE (VoLTE) and Voice over Wi-Fi (VoWiFi) are IP-based voice technologies that replace the legacy
circuit-switched voice channel used in 2G and 3G networks.
VoLTE: With VoLTE, voice calls 

--- Chunk 2 ---
voice simultaneously without degradation. VoLTE requires a compatible device, a VoLTE-enabled SIM, and an
account that has VoLTE activated.
Enabling VoLTE: On most Android devices navigate to Settings > Mobile Network > VoLTE and toggle it on. On
iPhone go to Settings > Mobile Data > Mobile Data Opt

--- Chunk 3 ---
prioritised over general data traffic. This prevents voice quality degradation during periods of network congestion.
Without QoS, voice packets would compete with video streaming and file downloads, causing jitter and packet
loss.
Fallback Behaviour: If a VoLTE call cannot be established  - for exam



In [11]:
SYSTEM_PROMPT = """\
You are a helpful telecom assistant.
Answer the question using ONLY the context provided below.
If the context does not contain enough information, say so clearly.

Context:
{context}
"""

In [14]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_groq import ChatGroq

# --- Helper: join retrieved chunks into a single context string ---
def format_docs(docs):
    return "\n\n---\n\n".join(doc.page_content for doc in docs)

In [15]:
prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    ("human", "{question}"),
])

# --- LLM via Groq API ---
llm = ChatGroq(
    model="qwen/qwen3-32b",
    temperature=0,
    reasoning_format="parsed",
)

# --- Assemble the chain ---
chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

In [16]:
question = "How does international roaming work and what charges should I expect?"

print(f"Q: {question}\n")
print("A:", chain.invoke(question))

Q: How does international roaming work and what charges should I expect?

A: International roaming allows your device to connect to partner networks in visited countries when outside your home network's coverage. Here’s how it works and the charges you may incur:

### **How It Works**  
1. **Authentication**: The visited network verifies your subscription via signaling protocols (SS7/Diameter), and your home network authorizes service.  
2. **Traffic Handling**: All data, voice, and SMS traffic is routed back to your home network for billing, which may introduce latency.  

### **Charges**  
- **Roaming Zones**:  
  - **Zone A** (EU, UK, Australia, New Zealand): Lowest rates.  
  - **Zone B** (USA, Canada, Japan, Singapore): Moderate rates.  
  - **Zone C** (Rest of the world): Highest per-MB and per-minute charges.  
- **Billing Notes**:  
  - Purchase a roaming bundle before traveling to Zone B/C to avoid high standard rates.  
  - Pre-bundle data usage is charged at standard rates, 